# 波斯湾船舶图口径复核（2026-10-21）

数据为仓库已发布 DuckDB 的 AIS 归档；只读查询，无上游取数、无数据修改。代码基线 `3bbceefacc09d2599171cd72eccbaf31e62129c1`。

LFS 指针：
```
version https://git-lfs.github.com/spec/v1
oid sha256:dc53be4958c60eb428a38b6d62e664f89b2a49962e054450a5bf9d0a006bdb11
size 172503040
```

粒度：每 MMSI、真实观测时间、来源一条 AIS 报文。图表按 UTC 观测日与水域对 MMSI 去重；同船同日重复报文和跨来源不会重复计数。它是当日采样所见船舶集合的大小，不是跨边界进出流量，也不是同一时刻在场存量。

10 月 15 日归档 10,688 条波斯湾报文，2,090 个不同 MMSI；观测时间仅覆盖该日部分时段。10 月 6、15、16、21 日分别为 1,692、2,090、2,016、702 个 MMSI，其余日无归档。缺测不得填零，也不能用采样量差异推断真实船流或存量涨跌。MMSI 去重不等于核验了实体船舶身份或普查覆盖。

截图横轴使用小时格式与单一日期偏移，无法准确辨认各点所属日期。修复为完整 UTC 自然日期标签。后续用户明确要求切换存量/流量，新增独立口径：每日末次采样时点前120分钟有效船位存量，以及既有固定边界的已记录进入艘次；原并集算法仅保留作为本次复核证据，不再用于存量或流量图。


In [1]:
from pathlib import Path
import duckdb, json
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'runtime/monitoring.duckdb').exists())
query = "SELECT CAST(observed AT TIME ZONE 'UTC' AS DATE) AS observation_day,\n       COUNT(*) AS reports, COUNT(DISTINCT mmsi) AS distinct_mmsi,\n       MIN(observed) AS first_observation_utc,\n       MAX(observed) AS last_observation_utc,\n       COUNT(DISTINCT date_trunc('hour', observed)) AS hours_with_reports\nFROM ais_reports\nWHERE json_extract_string(payload, '$.region') = '波斯湾'\nGROUP BY 1 ORDER BY 1"
with duckdb.connect(str(root / 'runtime/monitoring.duckdb'), read_only=True) as conn:
    result = conn.execute(query)
    names = [c[0] for c in result.description]
    rows = [dict(zip(names, row)) for row in result.fetchall()]
print(json.dumps(rows, ensure_ascii=False, default=str, indent=2))


[
  {
    "observation_day": "2026-10-06",
    "reports": 5280,
    "distinct_mmsi": 1692,
    "first_observation_utc": "2026-10-06 12:00:39+00:00",
    "last_observation_utc": "2026-10-06 15:55:39+00:00",
    "hours_with_reports": 4
  },
  {
    "observation_day": "2026-10-15",
    "reports": 10688,
    "distinct_mmsi": 2090,
    "first_observation_utc": "2026-10-15 12:55:14+00:00",
    "last_observation_utc": "2026-10-15 20:35:31+00:00",
    "hours_with_reports": 9
  },
  {
    "observation_day": "2026-10-16",
    "reports": 9598,
    "distinct_mmsi": 2016,
    "first_observation_utc": "2026-10-16 02:36:26+00:00",
    "last_observation_utc": "2026-10-16 21:04:03+00:00",
    "hours_with_reports": 13
  },
  {
    "observation_day": "2026-10-21",
    "reports": 1231,
    "distinct_mmsi": 702,
    "first_observation_utc": "2026-10-21 12:20:43+00:00",
    "last_observation_utc": "2026-10-21 14:20:05+00:00",
    "hours_with_reports": 3
  }
]


In [2]:
from datetime import date
import vessel_charts, data_store
path = data_store.database_path()
history = vessel_charts.history(date(2026,10,21), (str(path), path.stat().st_mtime_ns))
flags = sorted({vessel_charts.flag_label(r) for r in history})
destinations = sorted({vessel_charts.destination_label(r) for r in history})
stocks = vessel_charts.stock_series(history, date(2026,10,6), date(2026,10,21), '日度', flags, destinations)
print(json.dumps([{k:r[k] for k in ('date','sample_at','波斯湾')} for r in stocks if r.get('sample_at')], ensure_ascii=False,default=str,indent=2))
events = vessel_charts.entry_history(date(2026,10,21), (str(path), path.stat().st_mtime_ns))
print('Available verified entry events:', len(events))


[
  {
    "date": "2026-10-06",
    "sample_at": "2026-10-06T15:55:39+00:00",
    "波斯湾": 1656
  },
  {
    "date": "2026-10-15",
    "sample_at": "2026-10-15T20:35:31+00:00",
    "波斯湾": 1654
  },
  {
    "date": "2026-10-16",
    "sample_at": "2026-10-16T21:04:03+00:00",
    "波斯湾": 1174
  },
  {
    "date": "2026-10-21",
    "sample_at": "2026-10-21T14:20:05+00:00",
    "波斯湾": 702
  }
]
Available verified entry events: 0
